In [29]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [31]:
from qiskit_aer import AerSimulator

backend = AerSimulator()

def quantum_random_bits(n):
    """Generate n random bits by measuring n qubits prepared in |+⟩ = H|0⟩."""
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)          # |0⟩ → 1/√2(|0⟩ + |1⟩)
    qc.measure(range(n), range(n))
    # Run directly, skip transpile to avoid the coupling-map qubit limit
    job = backend.run(qc, shots=1, memory=True)
    bits_str = job.result().get_memory()[0]
    # Qiskit stores qubit 0 at the rightmost character, so reverse
    return [int(b) for b in reversed(bits_str)]


In [32]:
N_BITS = 100  # number of qubits Alice transmits

# ALICE
alice_bits  = quantum_random_bits(N_BITS)   # secret bit values to encode
alice_bases = quantum_random_bits(N_BITS)   # 0 = S (standard), 1 = D (diagonal)

print("ALICE")
print(f"Bits  : {alice_bits[:]}")
print(f"Bases : {alice_bases[:]}")
print("  (basis 0 = S/standard: |0⟩,|1⟩   basis 1 = D/diagonal: |+⟩,|−⟩)")


ALICE
Bits  : [0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0]
Bases : [1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0]
  (basis 0 = S/standard: |0⟩,|1⟩   basis 1 = D/diagonal: |+⟩,|−⟩)


In [33]:
def transmit_qubit(alice_bit, alice_basis, bob_basis):
    """
    Simulate one qubit: Alice encodes, Bob measures.

    Encoding:
      Z basis (0): bit 0 → |0⟩,  bit 1 → |1⟩
      X basis (1): bit 0 → |+⟩,  bit 1 → |−⟩
    """
    qc = QuantumCircuit(1, 1)
    # Alice encodes her bit in her chosen basis
    if alice_bit == 1:
        qc.x(0)        # |0⟩ → |1⟩
    if alice_basis == 1:
        qc.h(0)        # switch to diagonal basis
    # Bob measures in his chosen basis
    if bob_basis == 1:
        qc.h(0)        # rotate back to standard before measuring
    qc.measure(0, 0)
    job = backend.run(qc, shots=1, memory=True)
    return int(job.result().get_memory()[0])

# BOB
bob_bases   = quantum_random_bits(N_BITS)   # 0 = S, 1 = D
bob_results = [transmit_qubit(alice_bits[i], alice_bases[i], bob_bases[i])
               for i in range(N_BITS)]

print("BOB")
print(f"Bases   : {bob_bases[:]}")
print(f"Results : {bob_results[:]}")


BOB
Bases   : [0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0]
Results : [1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]


In [34]:
# SIFTING
# Alice and Bob publicly announce their bases.
# They keep only the bits where they happened to use the same basis.
print("SIFTING (public basis comparison)")

sifted_alice = []
sifted_bob   = []

for i in range(N_BITS):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

n_sifted = len(sifted_alice)
print(f"Matching bases: {n_sifted} / {N_BITS}  (roughly 50% expected)")
print(f"Alice sifted key : {sifted_alice[:]}")
print(f"Bob   sifted key : {sifted_bob[:]}")


SIFTING (public basis comparison)
Matching bases: 52 / 100  (roughly 50% expected)
Alice sifted key : [1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]
Bob   sifted key : [1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]


In [35]:
# ATTACK DETECTION
# Alice and Bob sacrifice a small sample of their sifted key to check for errors/attacker.
# All sample bits should match if there are no attackers.
# 10% error threshold to account for noise.
print("ATTACK DETECTION")

SAMPLE_SIZE = min(20, n_sifted)
errors     = sum(a != b for a, b in zip(sifted_alice[:SAMPLE_SIZE], sifted_bob[:SAMPLE_SIZE]))
error_rate = errors / SAMPLE_SIZE

print(f"Sample size: {SAMPLE_SIZE} bits")
print(f"Errors:      {errors}")
print(f"Error rate:  {error_rate:.1%}")

THRESHOLD = 0.10   # if >10% errors, eavesdropping suspected

if error_rate > THRESHOLD:
    print(f"\nWARNING: error rate {error_rate:.1%} exceeds threshold — possible attacker!")
else:
    print(f"\nChannel is secure (error rate {error_rate:.1%}, which is ≤ {THRESHOLD:.0%}).")
    remaining_key = sifted_alice[SAMPLE_SIZE:]
    print(f"\nShared secret key ({len(remaining_key)} bits):")
    print(''.join(str(b) for b in remaining_key))


ATTACK DETECTION
Sample size: 20 bits
Errors:      0
Error rate:  0.0%

Channel is secure (error rate 0.0%, which is ≤ 10%).

Shared secret key (32 bits):
10100000100111101011010100011000
